# **I. BakoScope**


> **Notebook : Data Modeling**
>
> Notebook ini berisi penerapan berbagai metode pemodelan time series, termasuk **Simple Average**, **Moving Average**, **Simple Exponential Smoothing**, **Holt-Linear**, **Holt-Winters**, serta pemodelan **ARIMA** dan **SARIMA** untuk memprediksi harga sembako.

---

## Tim Pengembang BakoScope

### **[BakoScope](https://huggingface.co/spaces/elangcergasp/bakoscope)** - Forecasting Harga Sembako

Kami adalah tim yang terdiri dari tiga profesional yang bekerja sama untuk memberikan solusi berbasis data dalam analisis harga komoditas sembako di Indonesia. 
Dengan menggunakan berbagai teknik pemodelan data dan eksplorasi analisis, kami berfokus untuk memberikan wawasan yang lebih mendalam terkait fluktuasi harga sembako. 
Dalam proyek ini, kami menerapkan berbagai metode smoothing seperti **Simple Average**, **Moving Average**, **Simple Exponential Smoothing**, **Holt-Linear**, **Holt-Winters**, serta pemodelan **ARIMA** dan **SARIMA**. 

Data yang kami gunakan diperoleh dari **[badanpangan.go.id](https://badanpangan.go.id)**, dan kami memodelkan setiap kombinasi komoditas dan provinsi untuk menghasilkan forecast yang lebih akurat. Kami juga melakukan eksplorasi data dengan memeriksa **seasonality**, **stationarity**, dan **ACF PACF** untuk mendapatkan pemahaman yang lebih baik mengenai pola data yang ada.

### Anggota Tim:

1. **Arcana Anggreliya Klau Rissa** – *Data Engineer*  
   Memimpin implementasi dan pengembangan workflow engineering menggunakan **Apache Airflow** untuk melakukan **scraping** data secara periodik dan menjalankan proses **ETL** (Extract, Transform, Load). Bertugas mengintegrasikan sistem dan memastikan alur data yang lancar untuk mempersiapkan data yang siap dianalisis dan dimodelkan.

2. **Aris Trisnawan** – *Data Analyst*  
   Bertanggung jawab atas eksplorasi dan pembersihan data. Memfokuskan pada analisis eksplorasi data (EDA) dan visualisasi time series untuk mengidentifikasi pola harga sembako. Menghasilkan grafik dan dashboard yang dapat digunakan untuk memantau fluktuasi harga serta memberikan pemahaman yang lebih jelas kepada pengguna akhir.

3. **Elang Cergas Pembrani** – *Data Scientist*  
   Mengembangkan dan menerapkan berbagai teknik pemodelan time series termasuk **ARIMA**, **SARIMA**, serta metode smoothing seperti **Simple Exponential Smoothing** dan **Holt-Winters**. Memimpin bagian pengujian model dan analisis terkait **seasonality**, **stationarity**, serta **ACF PACF** untuk setiap kombinasi komoditas dan provinsi.

### Hasil Utama Proyek:

- **Analisis EDA pada Time Series Data Harga Sembako**  
  Kami melakukan eksplorasi data mendalam untuk mempelajari karakteristik dan pola harga sembako yang diperoleh dari **badanpangan.go.id**. Temuan utama terkait tren musiman dan fluktuasi harga kami rangkum dalam berbagai visualisasi yang mempermudah pemahaman.

- **Model Forecasting untuk Setiap Kombinasi Komoditas dan Provinsi**  
  Model prediksi harga sembako dikembangkan untuk setiap kombinasi komoditas dan provinsi, yang semuanya dapat diakses secara terintegrasi melalui **dashboard HuggingFace**. Hal ini memungkinkan pengguna untuk melihat dan memantau ramalan harga secara real-time.

- **Workflow Engineering untuk Data Scraping dan ETL**  
  Dengan menggunakan **Apache Airflow**, kami membangun pipeline otomatis untuk melakukan scraping data secara periodik dan proses ETL. Data yang diambil akan diproses menjadi dataset yang siap dianalisis dan dimodelkan, menjamin sistem berjalan dengan efisien dan terkini.

### Footnote: Batasan Data

Data yang digunakan dalam proyek ini terbatas pada **komoditas** dan **provinsi** berikut:

| **Komoditas**              | **Provinsi**           |
|----------------------------|------------------------|
| Beras Medium               | DKI Jakarta            |
| Daging Ayam Ras            | Jawa Barat             |
| Telur Ayam Ras             | Jawa Tengah            |
| Minyak Goreng Kemasan      | D.I Yogyakarta         |
|                            | Jawa Timur             |
|                            | Banten                 |


---

# **II. Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing, Holt
import pylab
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima.model import ARIMA, ARIMAResults
from statsmodels.tsa.statespace.sarimax import SARIMAX, SARIMAXResults
from sklearn.preprocessing import StandardScaler
from scipy.signal import find_peaks
import os
import dill as pickle
import json
import copy
import re
import warnings
from my_models import ModelWrapper, FixedValueModel, MovingAverageModel
warnings.filterwarnings('ignore')

# **III. Data Loading**

## **A. Import Data**

In [2]:
with open('./komoditas_id.json', 'r') as f:
    komoditas_ids_arr_all = json.load(f)
    display(komoditas_ids_arr_all)

with open('./provinsi_id.json', 'r') as f:
    provinsi_ids_arr_all = json.load(f)
    display(provinsi_ids_arr_all)


{'28': 'Beras Medium',
 '35': 'Daging Ayam Ras',
 '36': 'Telur Ayam Ras',
 '38': 'Minyak Goreng Kemasan',
 '152': 'Daging Kerbau Segar (Lokal)',
 '149': 'Daging Kerbau Beku (Impor Luar Negeri)',
 '127': 'Minyakita',
 '109': 'Beras SPHP',
 '108': 'Tepung Terigu Kemasan',
 '106': 'Ikan Bandeng',
 '105': 'Ikan Tongkol',
 '104': 'Ikan Kembung',
 '102': 'Jagung Tk Peternak',
 '101': 'Minyak Goreng Curah',
 '29': 'Kedelai Biji Kering (Impor)',
 '40': 'Tepung Terigu (Curah)',
 '27': 'Beras Premium',
 '107': 'Garam Konsumsi',
 '126': 'Cabai Merah Besar',
 '32': 'Cabai Merah Keriting',
 '33': 'Cabai Rawit Merah',
 '34': 'Daging Sapi Murni',
 '31': 'Bawang Putih Bonggol',
 '30': 'Bawang Merah',
 '37': 'Gula Konsumsi'}

{'1': 'Aceh',
 '2': 'Sumatera Utara',
 '3': 'Sumatera Barat',
 '4': 'Riau',
 '5': 'Jambi',
 '6': 'Sumatera Selatan',
 '7': 'Bengkulu',
 '8': 'Lampung',
 '9': 'Kepulauan Bangka Belitung',
 '10': 'Kepulauan Riau',
 '11': 'DKI Jakarta',
 '12': 'Jawa Barat',
 '13': 'Jawa Tengah',
 '14': 'D.I Yogyakarta',
 '15': 'Jawa Timur',
 '16': 'Banten',
 '17': 'Bali',
 '18': 'Nusa Tenggara Barat',
 '19': 'Nusa Tenggara Timur',
 '20': 'Kalimantan Barat',
 '21': 'Kalimantan Tengah',
 '22': 'Kalimantan Selatan',
 '23': 'Kalimantan Timur',
 '24': 'Kalimantan Utara',
 '25': 'Sulawesi Utara',
 '26': 'Sulawesi Tengah',
 '27': 'Sulawesi Selatan',
 '28': 'Sulawesi Tenggara',
 '29': 'Gorontalo',
 '30': 'Sulawesi Barat',
 '31': 'Maluku',
 '32': 'Maluku Utara',
 '33': 'Papua Barat',
 '34': 'Papua',
 '35': 'Papua Barat Daya',
 '36': 'Papua Pegunungan',
 '37': 'Papua Tengah',
 '38': 'Papua Selatan'}

In [3]:
data_file = './data_clean_interpolasi.csv'

df = pd.read_csv(data_file)

df

,date,komoditas,provinsi,harga
0,2021-02-18,Beras Medium,Jawa Timur,10600.000000
1,2021-02-19,Beras Medium,Jawa Timur,10594.736842
2,2021-02-20,Beras Medium,Jawa Timur,10589.473684
3,2021-02-21,Beras Medium,Jawa Timur,10584.210526
4,2021-02-22,Beras Medium,Jawa Timur,10578.947368
...,...,...,...,...
37435,2025-06-09,Minyak Goreng Kemasan,Jawa Timur,19712.000000
37436,2025-06-10,Minyak Goreng Kemasan,Jawa Timur,19885.000000
37437,2025-06-11,Minyak Goreng Kemasan,Jawa Timur,19785.000000
37438,2025-06-12,Minyak Goreng Kemasan,Jawa Timur,19744.000000


## **B. Preview Sample Data**

In [4]:
df.head(5)

,date,komoditas,provinsi,harga
0,2021-02-18,Beras Medium,Jawa Timur,10600.000000
1,2021-02-19,Beras Medium,Jawa Timur,10594.736842
2,2021-02-20,Beras Medium,Jawa Timur,10589.473684
3,2021-02-21,Beras Medium,Jawa Timur,10584.210526
4,2021-02-22,Beras Medium,Jawa Timur,10578.947368


In [5]:
df.tail(5)

,date,komoditas,provinsi,harga
37435,2025-06-09,Minyak Goreng Kemasan,Jawa Timur,19712.0
37436,2025-06-10,Minyak Goreng Kemasan,Jawa Timur,19885.0
37437,2025-06-11,Minyak Goreng Kemasan,Jawa Timur,19785.0
37438,2025-06-12,Minyak Goreng Kemasan,Jawa Timur,19744.0
37439,2025-06-13,Minyak Goreng Kemasan,Jawa Timur,19895.0


## **C. Informasi Awal**

### **Jumlah Baris dan Kolom**

In [6]:
print('Jumlah baris:', df.shape[0])
print('Jumlah kolom:', df.shape[1])

Jumlah baris: 37440
Jumlah kolom: 4


### **Summary Kolom**

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37440 entries, 0 to 37439
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       37440 non-null  object 
 1   komoditas  37440 non-null  object 
 2   provinsi   37440 non-null  object 
 3   harga      37440 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.1+ MB


### **Missing Value**

In [8]:
pd.DataFrame({'null_count': df.isnull().sum()})

,null_count
date,0
komoditas,0
provinsi,0
harga,0


# **D. Data Preprocessing**

## **1. Prepare Date As Index**

In [9]:
df_clean_1 = df.copy()

df_clean_1['date'] = pd.to_datetime(df_clean_1['date'])

df_clean_1 = df_clean_1.set_index('date').sort_index()

In [10]:
df_clean_1.head(5)

,komoditas,provinsi,harga
date,,,
2021-02-18,Beras Medium,Jawa Timur,10600.0
2021-02-18,Minyak Goreng Kemasan,Jawa Timur,14000.0
2021-02-18,Telur Ayam Ras,Jawa Timur,22500.0
2021-02-18,Daging Ayam Ras,Jawa Timur,34500.0
2021-02-19,Minyak Goreng Kemasan,Jawa Timur,14000.0


### **2. Group By Komoditas & Provinsi**

In [11]:
df_group_by_cols = ['komoditas', 'provinsi']
df_clean_2 = df_clean_1.groupby(by=df_group_by_cols)

In [12]:
df_clean_2.get_group(list(df_clean_2.groups.keys())[0]).head(5)

,komoditas,provinsi,harga
date,,,
2021-03-10,Beras Medium,Banten,9000.0
2021-03-11,Beras Medium,Banten,10000.0
2021-03-12,Beras Medium,Banten,9500.0
2021-03-13,Beras Medium,Banten,9000.0
2021-03-14,Beras Medium,Banten,9750.0


### **3. Get Only Harga Inside Group By**

In [13]:
col_target = 'harga'
df_clean_3 = df_clean_2[[col_target]]

In [14]:
df_clean_3.get_group(list(df_clean_3.groups.keys())[0]).head(5)

,harga
date,
2021-03-10,9000.0
2021-03-11,10000.0
2021-03-12,9500.0
2021-03-13,9000.0
2021-03-14,9750.0


### **4. Hasil Akhir Data Preprocessing**

In [15]:
df_clean_groups = df_clean_3

df_clean_groups.groups

{('Beras Medium', 'Banten'): [2021-03-10 00:00:00, 2021-03-11 00:00:00, 2021-03-12 00:00:00, 2021-03-13 00:00:00, 2021-03-14 00:00:00, 2021-03-15 00:00:00, 2021-03-16 00:00:00, 2021-03-17 00:00:00, 2021-03-18 00:00:00, 2021-03-19 00:00:00, 2021-03-20 00:00:00, 2021-03-21 00:00:00, 2021-03-22 00:00:00, 2021-03-23 00:00:00, 2021-03-24 00:00:00, 2021-03-25 00:00:00, 2021-03-26 00:00:00, 2021-03-27 00:00:00, 2021-03-28 00:00:00, 2021-03-29 00:00:00, 2021-03-30 00:00:00, 2021-03-31 00:00:00, 2021-04-01 00:00:00, 2021-04-02 00:00:00, 2021-04-03 00:00:00, 2021-04-04 00:00:00, 2021-04-05 00:00:00, 2021-04-06 00:00:00, 2021-04-07 00:00:00, 2021-04-08 00:00:00, 2021-04-09 00:00:00, 2021-04-10 00:00:00, 2021-04-11 00:00:00, 2021-04-12 00:00:00, 2021-04-13 00:00:00, 2021-04-14 00:00:00, 2021-04-15 00:00:00, 2021-04-16 00:00:00, 2021-04-17 00:00:00, 2021-04-18 00:00:00, 2021-04-19 00:00:00, 2021-04-20 00:00:00, 2021-04-21 00:00:00, 2021-04-22 00:00:00, 2021-04-23 00:00:00, 2021-04-24 00:00:00, 2021

# **IV. Exploratory Data Analysis**

## **A. Split Train and Test**

In [16]:
train_test_ratio = 0.9
# test_len = 90

modeling_data_groups = {}
for g, df_group in df_clean_groups.groups.items():
    print('Group:', g)
    tmp_train_len = int(df_group.shape[0] * train_test_ratio)
    # tmp_train_len = int(df_group.shape[0] - test_len)

    tmp_train = df_clean_groups.get_group(g)[:tmp_train_len]
    tmp_test = df_clean_groups.get_group(g)[tmp_train_len:]
    
    modeling_data_groups[g] = {
        'train': tmp_train,
        'test': tmp_test,
    }

    print('  Jumlah Train :', tmp_train.shape[0])
    print('  Jumlah Test  :', tmp_test.shape[0])

    print(f'  Range Date Train : [{tmp_train.iloc[0].name}] s/d [{tmp_train.iloc[-1].name}]')
    print(f'  Range Date Test  : [{tmp_test.iloc[0].name}] s/d [{tmp_test.iloc[-1].name}]')


Group: ('Beras Medium', 'Banten')
  Jumlah Train : 1401
  Jumlah Test  : 156
  Range Date Train : [2021-03-10 00:00:00] s/d [2025-01-08 00:00:00]
  Range Date Test  : [2025-01-09 00:00:00] s/d [2025-06-13 00:00:00]
Group: ('Beras Medium', 'D.I Yogyakarta')
  Jumlah Train : 1401
  Jumlah Test  : 156
  Range Date Train : [2021-03-10 00:00:00] s/d [2025-01-08 00:00:00]
  Range Date Test  : [2025-01-09 00:00:00] s/d [2025-06-13 00:00:00]
Group: ('Beras Medium', 'DKI Jakarta')
  Jumlah Train : 1395
  Jumlah Test  : 155
  Range Date Train : [2021-03-17 00:00:00] s/d [2025-01-09 00:00:00]
  Range Date Test  : [2025-01-10 00:00:00] s/d [2025-06-13 00:00:00]
Group: ('Beras Medium', 'Jawa Barat')
  Jumlah Train : 1404
  Jumlah Test  : 157
  Range Date Train : [2021-03-06 00:00:00] s/d [2025-01-07 00:00:00]
  Range Date Test  : [2025-01-08 00:00:00] s/d [2025-06-13 00:00:00]
Group: ('Beras Medium', 'Jawa Tengah')
  Jumlah Train : 1402
  Jumlah Test  : 156
  Range Date Train : [2021-03-09 00:00:00

## **B. Stationarity (Differencing Param - `param_d`)**

In [17]:
def check_stationarity(series):
    # Copied from https://machinelearningmastery.com/time-series-data-stationary-python/

    result = adfuller(series.values)

    print('ADF Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print('\t%s: %.3f' % (key, value))

    if (result[1] <= 0.05) & (result[4]['5%'] > result[0]):
        print("\u001b[32mStationary\u001b[0m")
        return True
    else:
        print("\x1b[31mNon-stationary\x1b[0m")
        return False


In [18]:
differencing_orders = {}

for g, modeling_data in modeling_data_groups.items():
    print('Group:', g)
    tmp_diff = modeling_data['train'][col_target]
    tmp_is_stationary = check_stationarity(tmp_diff)
    
    tmp_param_d = 0
    while not tmp_is_stationary:
        tmp_param_d += 1
        print('  Differencing:', tmp_param_d)
        tmp_diff = tmp_diff.diff().dropna()
        tmp_is_stationary = check_stationarity(tmp_diff)
    differencing_orders[g] = tmp_param_d
    print('  Order of Differencing:', tmp_param_d)
    
    print()

differencing_orders = pd.DataFrame({'Differencing_Order': differencing_orders})

Group: ('Beras Medium', 'Banten')
ADF Statistic: -0.769862
p-value: 0.827815
Critical Values:
	1%: -3.435
	5%: -2.864
	10%: -2.568
Non-stationary
  Differencing: 1
ADF Statistic: -18.399845
p-value: 0.000000
Critical Values:
	1%: -3.435
	5%: -2.864
	10%: -2.568
Stationary
  Order of Differencing: 1

Group: ('Beras Medium', 'D.I Yogyakarta')
ADF Statistic: -0.874542
p-value: 0.796241
Critical Values:
	1%: -3.435
	5%: -2.864
	10%: -2.568
Non-stationary
  Differencing: 1
ADF Statistic: -6.212948
p-value: 0.000000
Critical Values:
	1%: -3.435
	5%: -2.864
	10%: -2.568
Stationary
  Order of Differencing: 1

Group: ('Beras Medium', 'DKI Jakarta')
ADF Statistic: -0.496174
p-value: 0.892761
Critical Values:
	1%: -3.435
	5%: -2.864
	10%: -2.568
Non-stationary
  Differencing: 1
ADF Statistic: -22.212178
p-value: 0.000000
Critical Values:
	1%: -3.435
	5%: -2.864
	10%: -2.568
Stationary
  Order of Differencing: 1

Group: ('Beras Medium', 'Jawa Barat')
ADF Statistic: -1.111406
p-value: 0.710502
Crit

In [19]:
model_params = pd.DataFrame()
model_params['param_d'] = differencing_orders['Differencing_Order']

model_params

param_d
Beras Medium          Banten                1
                      D.I Yogyakarta        1
                      DKI Jakarta           1
                      Jawa Barat            1
                      Jawa Tengah           1
                      Jawa Timur            1
Daging Ayam Ras       Banten                0
                      D.I Yogyakarta        0
                      DKI Jakarta           0
                      Jawa Barat            0
                      Jawa Tengah           0
                      Jawa Timur            0
Minyak Goreng Kemasan Banten                1
                      D.I Yogyakarta        1
                      DKI Jakarta           1
                      Jawa Barat            1
                      Jawa Tengah           1
                      Jawa Timur            1
Telur Ayam Ras        Banten                0
                      D.I Yogyakarta        0
                      DKI Jakarta           0
                      Jawa Barat            1
                      Jawa Tengah           1
                      Jawa Timur            0

## **C. Seasonality (Seasonality Param - `param_s`)**

In [20]:
seasonal_params = {}

for g, modeling_data in modeling_data_groups.items():
    res1 = seasonal_decompose(modeling_data['train'][col_target].rename(str(g) + ' Additive'), model='additive')
    res2 = seasonal_decompose(modeling_data['train'][col_target].rename(str(g) + ' Multiplicative'), model='multiplicative')
    peakrange = min(len(res1.seasonal),len(res2.seasonal))
    
    peak1 = find_peaks(res1.seasonal.iloc[:peakrange])
    peak11 = find_peaks(res1.seasonal.iloc[peak1[0]])
    
    peak2 = find_peaks(res2.seasonal.iloc[:peakrange])
    peak22 = find_peaks(res2.seasonal.iloc[peak2[0]])
    
    scaled1 = pd.DataFrame(StandardScaler().fit_transform(pd.DataFrame(res1.resid.dropna()))).iloc[:, 0]
    scaled1.index = res1.resid.dropna().index
    mean1 = 0
    mean11 = scaled1.apply(lambda x: max(x,mean1)-min(x,mean1)).mean()
    mean111 = abs(mean1 - mean11)
    
    scaled2 = pd.DataFrame(StandardScaler().fit_transform(pd.DataFrame(res2.resid.dropna()))).iloc[:, 0]
    scaled2.index = res2.resid.dropna().index
    mean2 = 1
    mean22 = scaled2.apply(lambda x: max(x,mean2)-min(x,mean2)).mean()
    mean222 = abs(mean2 - mean22)
    
    tmp_param_s = {
        'Additive': round(peakrange/len(peak1[0])) if len(peak11[0]) == 0 else round(peakrange/len(peak11[0])),
        'Multiplicative': round(peakrange/len(peak2[0])) if len(peak22[0]) == 0 else round(peakrange/len(peak22[0])),
    }
    
    seasonal_params[g] = { 'Method': 'Additive' if mean111 < mean222 else 'Multiplicative' }
    seasonal_params[g]['Seasonality'] = tmp_param_s[seasonal_params[g]['Method']]
    
    # pylab.rcParams['figure.figsize'] = (14, 5)
    # ax=res1.plot()
    # plt.show()
    
    # pylab.rcParams['figure.figsize'] = (14, 5)
    # ax=res2.plot()
    # plt.show()
    
    # print(f'[{g}] Additive Peak In {peakrange} days : {len(peak1[0])} => {seasonal_params[g]}')
    # print(f'{res1.seasonal.iloc[peak1[0]]}')
    # print(f'{res1.seasonal.iloc[pd.Series(peak1[0]).iloc[peak11[0]]]}')
    # print(f'[{g}] Multiplc Peak In {peakrange} days : {len(peak2[0])} => {seasonal_params[g]}')
    # print(f'{res2.seasonal.iloc[peak2[0]]}')
    # print(f'{res2.seasonal.iloc[pd.Series(peak2[0]).iloc[peak22[0]]]}')
    # print('-'*80)

seasonal_params = pd.DataFrame(seasonal_params).T

display(seasonal_params)


Method Seasonality
Beras Medium          Banten          Multiplicative           7
                      D.I Yogyakarta  Multiplicative           7
                      DKI Jakarta     Multiplicative           7
                      Jawa Barat      Multiplicative           7
                      Jawa Tengah     Multiplicative           7
                      Jawa Timur      Multiplicative           7
Daging Ayam Ras       Banten          Multiplicative           7
                      D.I Yogyakarta  Multiplicative           7
                      DKI Jakarta     Multiplicative           7
                      Jawa Barat      Multiplicative           7
                      Jawa Tengah     Multiplicative           7
                      Jawa Timur      Multiplicative           7
Minyak Goreng Kemasan Banten          Multiplicative           7
                      D.I Yogyakarta  Multiplicative           7
                      DKI Jakarta     Multiplicative           7
                      Jawa Barat      Multiplicative           7
                      Jawa Tengah     Multiplicative           7
                      Jawa Timur      Multiplicative           7
Telur Ayam Ras        Banten          Multiplicative           7
                      D.I Yogyakarta  Multiplicative           7
                      DKI Jakarta     Multiplicative           7
                      Jawa Barat      Multiplicative           7
                      Jawa Tengah     Multiplicative           7
                      Jawa Timur      Multiplicative           7

In [21]:
model_params['param_s'] = seasonal_params['Seasonality']
model_params['method'] = seasonal_params['Method'].str.lower()

model_params

param_d param_s          method
Beras Medium          Banten                1       7  multiplicative
                      D.I Yogyakarta        1       7  multiplicative
                      DKI Jakarta           1       7  multiplicative
                      Jawa Barat            1       7  multiplicative
                      Jawa Tengah           1       7  multiplicative
                      Jawa Timur            1       7  multiplicative
Daging Ayam Ras       Banten                0       7  multiplicative
                      D.I Yogyakarta        0       7  multiplicative
                      DKI Jakarta           0       7  multiplicative
                      Jawa Barat            0       7  multiplicative
                      Jawa Tengah           0       7  multiplicative
                      Jawa Timur            0       7  multiplicative
Minyak Goreng Kemasan Banten                1       7  multiplicative
                      D.I Yogyakarta        1       7  multiplicative
                      DKI Jakarta           1       7  multiplicative
                      Jawa Barat            1       7  multiplicative
                      Jawa Tengah           1       7  multiplicative
                      Jawa Timur            1       7  multiplicative
Telur Ayam Ras        Banten                0       7  multiplicative
                      D.I Yogyakarta        0       7  multiplicative
                      DKI Jakarta           0       7  multiplicative
                      Jawa Barat            1       7  multiplicative
                      Jawa Tengah           1       7  multiplicative
                      Jawa Timur            0       7  multiplicative

## **D. ACF & PACF**

### **1. PACF**

In [22]:
def compute_pacf(data, nlags=100):
    # Compute PACF
    lag_pacf = pacf(data, nlags=nlags)

    # Define a cutoff threshold (e.g., 1.96/sqrt(n) for 95% CI)
    n = len(data)
    cutoff_threshold = 1.96 / np.sqrt(n)

    # Find the cutoff index for PACF
    cutoff_index = np.where(np.abs(lag_pacf) < cutoff_threshold)[0]

    # Determine p based on the cutoff index
    if len(cutoff_index) > 0:
        p = cutoff_index[0]  # First index where PACF drops below the threshold
    else:
        p = len(lag_pacf) - 1  # Default to the maximum lag if no cutoff is found

    return p


In [23]:
pacfs = {}

for g, modeling_data in modeling_data_groups.items():
    param_p = compute_pacf(modeling_data['train'])
    pacfs[g] = param_p

pacfs = pd.DataFrame({'PACF': pacfs})

# pacfs

In [24]:
model_params['param_p'] = pacfs['PACF']

model_params

param_d param_s          method  param_p
Beras Medium          Banten                1       7  multiplicative        4
                      D.I Yogyakarta        1       7  multiplicative        6
                      DKI Jakarta           1       7  multiplicative        7
                      Jawa Barat            1       7  multiplicative        5
                      Jawa Tengah           1       7  multiplicative        5
                      Jawa Timur            1       7  multiplicative        5
Daging Ayam Ras       Banten                0       7  multiplicative        4
                      D.I Yogyakarta        0       7  multiplicative        5
                      DKI Jakarta           0       7  multiplicative        6
                      Jawa Barat            0       7  multiplicative        5
                      Jawa Tengah           0       7  multiplicative        5
                      Jawa Timur            0       7  multiplicative        7
Minyak Goreng Kemasan Banten                1       7  multiplicative        3
                      D.I Yogyakarta        1       7  multiplicative        3
                      DKI Jakarta           1       7  multiplicative        4
                      Jawa Barat            1       7  multiplicative        4
                      Jawa Tengah           1       7  multiplicative        3
                      Jawa Timur            1       7  multiplicative        3
Telur Ayam Ras        Banten                0       7  multiplicative        4
                      D.I Yogyakarta        0       7  multiplicative        3
                      DKI Jakarta           0       7  multiplicative        5
                      Jawa Barat            1       7  multiplicative        4
                      Jawa Tengah           1       7  multiplicative        8
                      Jawa Timur            0       7  multiplicative        6

In [25]:
def find_pacf_cutoff(series, alpha=0.05, nlags=40, verbose=False):
    """
    Finds the cutoff lag in the PACF like it would appear in a plot:
    the last lag with statistically significant partial autocorrelation.

    Parameters:
    - series (array-like): Time series data.
    - alpha (float): Significance level for confidence intervals (default 0.05 for 95% CI).
    - nlags (int): Number of lags to compute.
    - verbose (bool): If True, print detailed results.

    Returns:
    - cutoff (int): Last significant PACF lag.
    - pacf_vals (np.ndarray): PACF values.
    - confint (np.ndarray): Confidence intervals for each lag.
    """
    pacf_vals, confint = pacf(series, nlags=nlags, alpha=alpha, method='ywmle')
    
    significant_lags = []
    for lag in range(1, nlags + 1):  # skip lag 0
        lower, upper = confint[lag]
        if pacf_vals[lag] < lower or pacf_vals[lag] > upper:
            significant_lags.append(lag)
            if verbose:
                print(f"Lag {lag}: PACF = {pacf_vals[lag]:.3f}, CI = ({lower:.3f}, {upper:.3f}) => SIGNIFICANT")
        elif verbose:
            print(f"Lag {lag}: PACF = {pacf_vals[lag]:.3f}, CI = ({lower:.3f}, {upper:.3f}) => not significant")
    
    if not significant_lags:
        return 0, pacf_vals, confint

    return max(significant_lags), pacf_vals, confint

In [26]:
pacfs2 = {}

for g, modeling_data in modeling_data_groups.items():
    param_p = compute_pacf(modeling_data['train'])
    pacfs2[g] = param_p

pacfs2 = pd.DataFrame({'PACF': pacfs2})

pacfs2

PACF
Beras Medium          Banten             4
                      D.I Yogyakarta     6
                      DKI Jakarta        7
                      Jawa Barat         5
                      Jawa Tengah        5
                      Jawa Timur         5
Daging Ayam Ras       Banten             4
                      D.I Yogyakarta     5
                      DKI Jakarta        6
                      Jawa Barat         5
                      Jawa Tengah        5
                      Jawa Timur         7
Minyak Goreng Kemasan Banten             3
                      D.I Yogyakarta     3
                      DKI Jakarta        4
                      Jawa Barat         4
                      Jawa Tengah        3
                      Jawa Timur         3
Telur Ayam Ras        Banten             4
                      D.I Yogyakarta     3
                      DKI Jakarta        5
                      Jawa Barat         4
                      Jawa Tengah        8
                      Jawa Timur         6

### **2. ACF**

In [27]:
def find_acf_plot_cutoff(data, alpha=0.05, nlags=40):
    """
    Finds the last significant lag in ACF, using symmetric confidence intervals
    centered at 0, just like plot_acf displays.

    Parameters:
    - data (array-like): Time series data
    - alpha (float): Significance level (default 0.05)
    - nlags (int): Number of lags to consider

    Returns:
    - cutoff (int): Last lag where ACF is statistically significant (outside CI)
    - acf_vals (np.ndarray): ACF values
    - ci_bounds (np.ndarray): ±CI bounds (symmetric) at each lag
    """
    # Get ACF and 95% confidence intervals
    acf_vals, confint = acf(data, nlags=nlags, alpha=alpha)
    
    # Symmetric CI bounds centered at 0 (like plot_acf's blue region)
    ci_bounds = confint[:, 1] - acf_vals  # upper - acf = +CI width
    
    # Identify significant lags
    significant = np.abs(acf_vals) > ci_bounds
    
    # Find last significant lag (excluding lag 0)
    significant[0] = False  # lag 0 always = 1, ignore
    for lag in range(1, nlags):
        if significant[lag-1] and not significant[lag]:
            return lag, acf_vals, ci_bounds

    # If never turns insignificant after being significant, return 0
    return 0, acf_vals, ci_bounds


In [28]:
acfs = {}

for g, modeling_data in modeling_data_groups.items():
    # param_q = compute_acf(modeling_data['train'][col_target], param_p=model_params.loc[g, 'param_p'], param_d=model_params.loc[g, 'param_d'])
    tmp_df = modeling_data['train'][col_target]
    tmp_d = model_params.loc[g, 'param_d']
    while tmp_d > 0:
        tmp_df = tmp_df.diff().dropna()
        tmp_d -= 1
    param_q, acf_vals, ci_bounds = find_acf_plot_cutoff(tmp_df, nlags=100)
    acfs[g] = param_q

acfs = pd.DataFrame({'ACF': acfs})

acfs

ACF
Beras Medium          Banten            2
                      D.I Yogyakarta    3
                      DKI Jakarta       2
                      Jawa Barat        2
                      Jawa Tengah       4
                      Jawa Timur        2
Daging Ayam Ras       Banten           43
                      D.I Yogyakarta   43
                      DKI Jakarta      82
                      Jawa Barat       45
                      Jawa Tengah      47
                      Jawa Timur       43
Minyak Goreng Kemasan Banten            2
                      D.I Yogyakarta    2
                      DKI Jakarta       2
                      Jawa Barat        2
                      Jawa Tengah       4
                      Jawa Timur        2
Telur Ayam Ras        Banten           75
                      D.I Yogyakarta   82
                      DKI Jakarta      67
                      Jawa Barat        2
                      Jawa Tengah       2
                      Jawa Timur       90

In [29]:
MAX_Q = 7

In [30]:
model_params['param_q'] = acfs['ACF']#.apply(lambda x: min(x, MAX_Q))

model_params

param_d param_s          method  \
Beras Medium          Banten                1       7  multiplicative   
                      D.I Yogyakarta        1       7  multiplicative   
                      DKI Jakarta           1       7  multiplicative   
                      Jawa Barat            1       7  multiplicative   
                      Jawa Tengah           1       7  multiplicative   
                      Jawa Timur            1       7  multiplicative   
Daging Ayam Ras       Banten                0       7  multiplicative   
                      D.I Yogyakarta        0       7  multiplicative   
                      DKI Jakarta           0       7  multiplicative   
                      Jawa Barat            0       7  multiplicative   
                      Jawa Tengah           0       7  multiplicative   
                      Jawa Timur            0       7  multiplicative   
Minyak Goreng Kemasan Banten                1       7  multiplicative   
                      D.I Yogyakarta        1       7  multiplicative   
                      DKI Jakarta           1       7  multiplicative   
                      Jawa Barat            1       7  multiplicative   
                      Jawa Tengah           1       7  multiplicative   
                      Jawa Timur            1       7  multiplicative   
Telur Ayam Ras        Banten                0       7  multiplicative   
                      D.I Yogyakarta        0       7  multiplicative   
                      DKI Jakarta           0       7  multiplicative   
                      Jawa Barat            1       7  multiplicative   
                      Jawa Tengah           1       7  multiplicative   
                      Jawa Timur            0       7  multiplicative   

                                      param_p  param_q  
Beras Medium          Banten                4        2  
                      D.I Yogyakarta        6        3  
                      DKI Jakarta           7        2  
                      Jawa Barat            5        2  
                      Jawa Tengah           5        4  
                      Jawa Timur            5        2  
Daging Ayam Ras       Banten                4       43  
                      D.I Yogyakarta        5       43  
                      DKI Jakarta           6       82  
                      Jawa Barat            5       45  
                      Jawa Tengah           5       47  
                      Jawa Timur            7       43  
Minyak Goreng Kemasan Banten                3        2  
                      D.I Yogyakarta        3        2  
                      DKI Jakarta           4        2  
                      Jawa Barat            4        2  
                      Jawa Tengah           3        4  
                      Jawa Timur            3        2  
Telur Ayam Ras        Banten                4       75  
                      D.I Yogyakarta        3       82  
                      DKI Jakarta           5       67  
                      Jawa Barat            4        2  
                      Jawa Tengah           8        2  
                      Jawa Timur            6       90

# **V. Data Modeling**

In [31]:
def hash_group(group):
    komoditas_name, provinsi_name = group

    # Reverse lookup komoditas ID
    komoditas_id = next((k for k, v in komoditas_ids_arr_all.items() if v == komoditas_name), None)
    if komoditas_id is None:
        raise ValueError(f"Unknown komoditas: '{komoditas_name}'")

    # Reverse lookup provinsi ID
    provinsi_id = next((k for k, v in provinsi_ids_arr_all.items() if v == provinsi_name), None)
    if provinsi_id is None:
        raise ValueError(f"Unknown provinsi: '{provinsi_name}'")

    return f"komoditas_{komoditas_id}_provinsi_{provinsi_id}"

def safe_filename(s):
    # Remove parentheses
    s = s.replace('(', '').replace(')', '')
    # Replace commas with underscore and remove spaces
    s = s.replace(',', '_').replace(' ', '')
    # Remove any other disallowed chars (keep only alphanumeric, underscore, dash)
    return re.sub(r'[^A-Za-z0-9_-]', '', s)

# MODEL MINIMIZER
def safe_minimize_model_copy(model_fit):
    # Deep copy to preserve original
    minimized_model = copy.deepcopy(model_fit)
    
    # Attributes to keep for minimal functionality
    keep_attrs = ['model', 'params', 'filter_results', 'data', 'k_exog', 'k_trend']
    
    for attr in list(vars(minimized_model)):
        if attr not in keep_attrs:
            delattr(minimized_model, attr)
    
    return minimized_model

def backup_old_model(old_path, prefix = ''):
    base, ext = os.path.splitext(old_path)
    backup_path = f"{base}_backup_{prefix}{ext}"
    if os.path.exists(backup_path):
        # Avoid overwrite by adding a numeric suffix
        idx = 1
        while os.path.exists(f"{base}_backup_{prefix}_{idx}{ext}"):
            idx += 1
        backup_path = f"{base}_backup_{prefix}_{idx}{ext}"
    os.rename(old_path, backup_path)
    print(f"Backed up old model to {backup_path}")


## **A. Smoothing**

In [32]:
def get_eval_smoothing(value, pred, title = None, echo = False):
    rmse = root_mean_squared_error(value, pred)
    mae = mean_absolute_error(value, pred)
    mape = mean_absolute_percentage_error(value, pred)

    subtitle = '' if title is None else f'[{title}] '
    if echo:
        print(f"{subtitle}MAE :", mae)
        print(f"{subtitle}MAPE :", mape)
        print(f"{subtitle}RMSE :", rmse)
    
    return pd.DataFrame({title: {
        'MAE': mae,
        'MAPE': mape,
        'RMSE': rmse,
    }}).T


In [33]:
test_predicts = {g: modeling_data_groups[g]['test'].copy() for g in modeling_data_groups.keys()}

In [34]:
smoothing_evals = pd.DataFrame()

### **1. Naive Approach**

In [35]:
# models_naive = {}

# for g, modeling_data in modeling_data_groups.items():
#     models_naive[g] = FixedValueModel(modeling_data['train'][col_target].iloc[-1], name='Naive')
#     tmp_smoothing = models_naive[g].predict()
#     test_predicts[g]['naive'] = tmp_smoothing


In [36]:
# for g, modeling_data in modeling_data_groups.items():
#     tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['naive'], title='Naive')
#     for i, c in enumerate(df_group_by_cols):
#         tmp_eval.insert(0, c, g[i])
#     smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

# display(smoothing_evals.loc['Naive'])

### **2. Simple Average**

In [37]:
models_simple_average = {}

for g, modeling_data in modeling_data_groups.items():
    models_simple_average[g] = FixedValueModel(modeling_data['train'][col_target].mean(), name='SimpleAverage')
    tmp_smoothing = models_simple_average[g].predict()
    test_predicts[g]['simple_average'] = tmp_smoothing


In [38]:
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['simple_average'], title='Simple Average')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Simple Average'])

,provinsi,komoditas,MAE,MAPE,RMSE
Simple Average,Banten,Beras Medium,1586.545956,0.123624,1588.786139
Simple Average,D.I Yogyakarta,Beras Medium,1866.604788,0.142816,1867.457273
Simple Average,DKI Jakarta,Beras Medium,1241.678495,0.095169,1249.397132
Simple Average,Jawa Barat,Beras Medium,1708.997911,0.130923,1710.330063
Simple Average,Jawa Tengah,Beras Medium,1731.759336,0.131708,1732.939682
Simple Average,Jawa Timur,Beras Medium,1707.790327,0.133606,1710.062906
Simple Average,Banten,Daging Ayam Ras,959.243809,0.026669,1244.115298
Simple Average,D.I Yogyakarta,Daging Ayam Ras,2127.498435,0.071084,3068.857882
Simple Average,DKI Jakarta,Daging Ayam Ras,723.288241,0.018701,976.088900
Simple Average,Jawa Barat,Daging Ayam Ras,1239.707911,0.035666,1556.258972


### **3. Moving Average**

In [39]:
models_moving_average = {}

for g, modeling_data in modeling_data_groups.items():
    tmp_model = MovingAverageModel(model_params.loc[g]['param_s'])
    tmp_model.fit(modeling_data['train'][col_target])
    models_moving_average[g] = tmp_model
    tmp_smoothing = models_moving_average[g].predict(modeling_data['test'][col_target])
    test_predicts[g]['moving_average'] = tmp_smoothing


Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with frequency: D
Model fitted successfully with f

In [40]:
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['moving_average'], title='Moving Average')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Moving Average'])

,provinsi,komoditas,MAE,MAPE,RMSE
Moving Average,Banten,Beras Medium,148.734711,0.011628,165.951554
Moving Average,D.I Yogyakarta,Beras Medium,76.186940,0.005812,94.491593
Moving Average,DKI Jakarta,Beras Medium,269.245085,0.020760,297.484209
Moving Average,Jawa Barat,Beras Medium,106.448628,0.008134,122.071382
Moving Average,Jawa Tengah,Beras Medium,89.028944,0.006752,105.922323
Moving Average,Jawa Timur,Beras Medium,100.823266,0.007855,126.509979
Moving Average,Banten,Daging Ayam Ras,1009.087791,0.028197,1360.285549
Moving Average,D.I Yogyakarta,Daging Ayam Ras,2595.845729,0.086312,3567.697851
Moving Average,DKI Jakarta,Daging Ayam Ras,646.872972,0.016928,782.196033
Moving Average,Jawa Barat,Daging Ayam Ras,1877.589708,0.054413,2316.245198


### **4. Simple Exponential Smoothing**

In [41]:
models_simple_exp_smoothing = {}

for g, modeling_data in modeling_data_groups.items():
    tmp_df = modeling_data['train'][col_target]
    tmp_d = model_params.loc[g, 'param_d']
    while tmp_d > 0:
        tmp_df = tmp_df.diff().dropna()
        tmp_d -= 1
    models_simple_exp_smoothing[g] = SimpleExpSmoothing(np.asarray(tmp_df)).fit(smoothing_level=0.2,optimized=False)
    tmp_smoothing = models_simple_exp_smoothing[g].forecast(len(modeling_data['test']))
    test_predicts[g]['simple_exp_smoothing'] = tmp_smoothing


In [42]:
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['simple_exp_smoothing'], title='Simple Exponential Smoothing')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Simple Exponential Smoothing'])

,provinsi,komoditas,MAE,MAPE,RMSE
Simple Exponential Smoothing,Banten,Beras Medium,12784.726851,0.996494,12785.005044
Simple Exponential Smoothing,D.I Yogyakarta,Beras Medium,13071.066633,1.000191,13071.188399
Simple Exponential Smoothing,DKI Jakarta,Beras Medium,13104.893989,1.005506,13105.627577
Simple Exponential Smoothing,Jawa Barat,Beras Medium,13022.052819,0.997770,13022.227715
Simple Exponential Smoothing,Jawa Tengah,Beras Medium,13139.364967,0.999461,13139.520587
Simple Exponential Smoothing,Jawa Timur,Beras Medium,12738.502539,0.996882,12738.807412
Simple Exponential Smoothing,Banten,Daging Ayam Ras,959.822860,0.026690,1246.909700
Simple Exponential Smoothing,D.I Yogyakarta,Daging Ayam Ras,2710.292354,0.089885,3655.840022
Simple Exponential Smoothing,DKI Jakarta,Daging Ayam Ras,702.713234,0.018502,820.148616
Simple Exponential Smoothing,Jawa Barat,Daging Ayam Ras,2007.409429,0.058108,2436.518669


### **5. Holt Linear Trend**

In [43]:
models_holt_linear = {}

for g, modeling_data in modeling_data_groups.items():
    # tmp_df = modeling_data['train'][col_target]
    # tmp_d = model_params.loc[g, 'param_d']
    # while tmp_d > 0:
    #     tmp_df = tmp_df.diff().dropna()
    #     tmp_d -= 1
    models_holt_linear[g] = Holt(modeling_data['train'][col_target]).fit(smoothing_level = 0.3,smoothing_slope = 0.1)
    tmp_smoothing = models_holt_linear[g].forecast(len(modeling_data['test']))
    test_predicts[g]['holt_linear'] = tmp_smoothing


In [44]:
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['holt_linear'], title='Holt Linear')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Holt Linear'])

,provinsi,komoditas,MAE,MAPE,RMSE
Holt Linear,Banten,Beras Medium,4000.018046,0.312340,4587.629713
Holt Linear,D.I Yogyakarta,Beras Medium,143.981597,0.011013,174.568409
Holt Linear,DKI Jakarta,Beras Medium,1987.693600,0.153051,2317.584615
Holt Linear,Jawa Barat,Beras Medium,908.994046,0.069750,1082.818500
Holt Linear,Jawa Tengah,Beras Medium,481.183036,0.036569,569.067579
Holt Linear,Jawa Timur,Beras Medium,1909.004249,0.149141,2202.986623
Holt Linear,Banten,Daging Ayam Ras,6098.303026,0.169567,7231.844837
Holt Linear,D.I Yogyakarta,Daging Ayam Ras,1248.335838,0.039521,1704.204603
Holt Linear,DKI Jakarta,Daging Ayam Ras,4607.105848,0.121176,5238.277357
Holt Linear,Jawa Barat,Daging Ayam Ras,1280.184738,0.035637,1650.362767


### **6. Holt-Winters**

In [45]:
models_holt_winter_additive = {}
models_holt_winter_multiplicative = {}

for g, modeling_data in modeling_data_groups.items():
    tmp_df = modeling_data['train'][col_target]
    tmp_d = model_params.loc[g, 'param_d']
    while tmp_d > 0:
        tmp_df = tmp_df.diff().dropna()
        tmp_d -= 1

    models_holt_winter_additive[g] = ExponentialSmoothing(np.asarray(tmp_df) ,seasonal_periods=model_params.loc[g, 'param_s'],trend='additive', seasonal='additive',).fit()
    tmp_smoothing = models_holt_winter_additive[g].forecast(len(modeling_data['test']))
    test_predicts[g]['holt_winter_additive'] = tmp_smoothing
    
    # models_holt_winter_multiplicative[g] = ExponentialSmoothing(np.asarray(tmp_df) ,seasonal_periods=model_params.loc[g, 'param_s'],trend='multiplicative', seasonal='multiplicative',).fit()
    # tmp_smoothing = models_holt_winter_multiplicative[g].forecast(len(modeling_data['test']))
    # test_predicts[g]['holt_winter_multiplicative'] = tmp_smoothing


In [46]:
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['holt_winter_additive'], title='Holt Winter - Additive')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()
    
    # tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['holt_winter_multiplicative'], title='Holt Winter - Multiplicative')
    # for i, c in enumerate(df_group_by_cols):
    #     tmp_eval.insert(0, c, g[i])
    # smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Holt Winter - Additive'])
# display(smoothing_evals.loc['Holt Winter - Multiplicative'])

,provinsi,komoditas,MAE,MAPE,RMSE
Holt Winter - Additive,Banten,Beras Medium,12583.442649,0.980777,12584.732112
Holt Winter - Additive,D.I Yogyakarta,Beras Medium,13061.086835,0.999428,13061.208298
Holt Winter - Additive,DKI Jakarta,Beras Medium,13073.575161,1.003113,13074.199106
Holt Winter - Additive,Jawa Barat,Beras Medium,13018.111524,0.997465,13018.399255
Holt Winter - Additive,Jawa Tengah,Beras Medium,13109.950609,0.997225,13110.319579
Holt Winter - Additive,Jawa Timur,Beras Medium,12775.214456,0.999755,12775.539113
Holt Winter - Additive,Banten,Daging Ayam Ras,1780.015959,0.049841,2359.034496
Holt Winter - Additive,D.I Yogyakarta,Daging Ayam Ras,2390.818771,0.079880,3407.027995
Holt Winter - Additive,DKI Jakarta,Daging Ayam Ras,3352.802226,0.088876,3989.616100
Holt Winter - Additive,Jawa Barat,Daging Ayam Ras,1778.460402,0.051569,2271.466131


### **Summary of Smoothing**

In [47]:
eval_cols = ['MAE', 'MAPE', 'RMSE']
model_eval_dfs = {}
for g, modeling_data in modeling_data_groups.items():
    # print('Group:', g)
    model_eval_dfs[g] = {}
    for ev in eval_cols:
        model_eval_dfs[g][ev] = smoothing_evals[(smoothing_evals['komoditas'] == g[0]) & (smoothing_evals['provinsi'] == g[1])][ev].idxmin()
        # print(f'Min of [{ev}] :', smoothing_evals[(smoothing_evals['komoditas'] == g[0]) & (smoothing_evals['provinsi'] == g[1])][ev].idxmin())
    # print()

model_eval_dfs = pd.DataFrame(model_eval_dfs).T
model_eval_dfs.insert(0, 'Best', model_eval_dfs[eval_cols].mode(axis=1))

model_eval_dfs

Best  \
Beras Medium          Banten                  Moving Average   
                      D.I Yogyakarta          Moving Average   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Moving Average   
                      Jawa Tengah             Moving Average   
                      Jawa Timur              Moving Average   
Daging Ayam Ras       Banten                  Simple Average   
                      D.I Yogyakarta             Holt Linear   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Simple Average   
                      Jawa Tengah     Holt Winter - Additive   
                      Jawa Timur              Simple Average   
Minyak Goreng Kemasan Banten                  Moving Average   
                      D.I Yogyakarta          Moving Average   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Moving Average   
                      Jawa Tengah             Moving Average   
                      Jawa Timur              Moving Average   
Telur Ayam Ras        Banten                  Simple Average   
                      D.I Yogyakarta          Simple Average   
                      DKI Jakarta             Simple Average   
                      Jawa Barat              Simple Average   
                      Jawa Tengah             Simple Average   
                      Jawa Timur              Simple Average   

                                                         MAE  \
Beras Medium          Banten                  Moving Average   
                      D.I Yogyakarta          Moving Average   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Moving Average   
                      Jawa Tengah             Moving Average   
                      Jawa Timur              Moving Average   
Daging Ayam Ras       Banten                  Simple Average   
                      D.I Yogyakarta             Holt Linear   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Simple Average   
                      Jawa Tengah     Holt Winter - Additive   
                      Jawa Timur              Simple Average   
Minyak Goreng Kemasan Banten                  Moving Average   
                      D.I Yogyakarta          Moving Average   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Moving Average   
                      Jawa Tengah             Moving Average   
                      Jawa Timur              Moving Average   
Telur Ayam Ras        Banten                  Simple Average   
                      D.I Yogyakarta          Simple Average   
                      DKI Jakarta             Simple Average   
                      Jawa Barat              Simple Average   
                      Jawa Tengah             Simple Average   
                      Jawa Timur              Simple Average   

                                                        MAPE  \
Beras Medium          Banten                  Moving Average   
                      D.I Yogyakarta          Moving Average   
                      DKI Jakarta             Moving Average   
                      Jawa Barat              Moving Average   
                      Jawa Tengah             Moving Average   
                      Jawa Timur              Moving Average   
Daging Ayam Ras       Banten                  Simple Average   
                      D.I Yogyakarta             Holt Linear   
                      DKI Jakarta             Moving Average   
                      Jawa Barat                 Holt Linear   
                      Jawa Tengah     Holt Winter - Additive   
                      Jawa Timur              Simple Average   
Minyak Goreng Kemasan 

## **B. Model ARIMA**

### **Model ARIMA Training**

In [48]:
def make_arima_filename(model_id, order):
    p, d, q = order
    return f"./models/model_arima_{model_id}_order_{p}_{d}_{q}.msgpack"

# === ARIMA MODEL TRAINING ===
models_arima = {}

for g, modeling_data in modeling_data_groups.items():
    model_id = hash_group(g)

    param_p = model_params.loc[g]['param_p']
    param_d = model_params.loc[g]['param_d']
    param_q = model_params.loc[g]['param_q']
    
    param_q = min(param_q, MAX_Q)
    
    order = (param_p, param_d, param_q)
    arima_file = make_arima_filename(model_id, order)

    retrain_arima = True
    if os.path.exists(arima_file):
        try:
            loaded_arima = ARIMAResults.load(arima_file)
            saved_order = loaded_arima.model.order
            if saved_order == order:
                models_arima[g] = loaded_arima
                print(f"[{g}] Loaded existing ARIMA model from {arima_file}")
                retrain_arima = False
            else:
                # Backup old file with old params in filename
                old_order_str = '_'.join(map(str, saved_order))
                backup_old_model(arima_file, old_order_str)
        except Exception as e:
            print(f"[{g}] Failed loading ARIMA model: {e}. Will retrain.")

    if retrain_arima:
        print(f"[{g}] Training ARIMA{order}...")
        try:
            model_arima = ARIMA(
                modeling_data['train'][col_target],
                order=order,
            ).fit()
        except np.linalg.LinAlgError as e:
            print(f"[{g}] LinAlgError caught: {e}. Retrying with relaxed constraints...")
            model_arima = ARIMA(
                modeling_data['train'][col_target],
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit()
        model_arima.save(arima_file)
        models_arima[g] = model_arima
        print(f"[{g}] ARIMA model trained and saved as {arima_file}")
        display(models_arima[g].summary())


[('Beras Medium', 'Banten')] Loaded existing ARIMA model from ./models/model_arima_komoditas_28_provinsi_16_order_4_1_2.msgpack
[('Beras Medium', 'D.I Yogyakarta')] Loaded existing ARIMA model from ./models/model_arima_komoditas_28_provinsi_14_order_6_1_3.msgpack
[('Beras Medium', 'DKI Jakarta')] Loaded existing ARIMA model from ./models/model_arima_komoditas_28_provinsi_11_order_7_1_2.msgpack
[('Beras Medium', 'Jawa Barat')] Loaded existing ARIMA model from ./models/model_arima_komoditas_28_provinsi_12_order_5_1_2.msgpack
[('Beras Medium', 'Jawa Tengah')] Loaded existing ARIMA model from ./models/model_arima_komoditas_28_provinsi_13_order_5_1_4.msgpack
[('Beras Medium', 'Jawa Timur')] Loaded existing ARIMA model from ./models/model_arima_komoditas_28_provinsi_15_order_5_1_2.msgpack
[('Daging Ayam Ras', 'Banten')] Loaded existing ARIMA model from ./models/model_arima_komoditas_35_provinsi_16_order_4_0_7.msgpack
[('Daging Ayam Ras', 'D.I Yogyakarta')] Loaded existing ARIMA model from ./

### **Model ARIMA Evaluation**

In [49]:
for g, modeling_data in modeling_data_groups.items():
    tmp_predict = models_arima[g].forecast(len(modeling_data['test']))
    test_predicts[g]['arima'] = tmp_predict


In [50]:
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['arima'], title='Model ARIMA')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Model ARIMA'])

,provinsi,komoditas,MAE,MAPE,RMSE
Model ARIMA,Banten,Beras Medium,84.348507,0.006596,105.183418
Model ARIMA,D.I Yogyakarta,Beras Medium,62.692953,0.004780,83.474617
Model ARIMA,DKI Jakarta,Beras Medium,151.255983,0.011634,167.267071
Model ARIMA,Jawa Barat,Beras Medium,81.901543,0.006287,101.085079
Model ARIMA,Jawa Tengah,Beras Medium,330.730631,0.025152,367.992753
Model ARIMA,Jawa Timur,Beras Medium,73.278240,0.005713,97.685103
Model ARIMA,Banten,Daging Ayam Ras,950.141291,0.026432,1239.852497
Model ARIMA,D.I Yogyakarta,Daging Ayam Ras,2268.154278,0.075742,3230.773287
Model ARIMA,DKI Jakarta,Daging Ayam Ras,581.479015,0.015150,764.069826
Model ARIMA,Jawa Barat,Daging Ayam Ras,1726.707476,0.050074,2163.528851


## **C. Model SARIMA**

### **Model SARIMA Training**

In [51]:

def find_seasonal_q(param_q, param_s):
    """
    Given the non-seasonal AR order (param_q) and seasonal period (param_s),
    determine the appropriate seasonal AR order (seasonal_q).
    
    This function assumes that seasonal_q is typically between 0 and param_s-1.
    The function avoids selecting seasonal_q that would conflict with the 
    non-seasonal AR lags.
    """
    # Try to select seasonal_q from 1 to param_s-1, avoiding 0 as it doesn't add much value for seasonal AR
    for seasonal_q in range(1, param_s):
        # Avoid seasonal AR lags that might conflict with the non-seasonal AR lags
        if seasonal_q <= param_q:
            continue  # Skip if seasonal_q is smaller than or equal to param_q
        
        # If no conflicts, return the seasonal_q
        return seasonal_q
    
    # If no valid seasonal_q was found, return 0 as a fallback
    return 0

def make_sarima_filename(model_id, order, seasonal_order):
    p, d, q = order
    sp, sd, sq, s = seasonal_order
    return f"./models/model_sarima_{model_id}_order_{p}_{d}_{q}_seasonal_{sp}_{sd}_{sq}_{s}.msgpack"


In [52]:
# %%script false --no-raise-error
# class MinimalSARIMAXModel:
#     def __init__(self, full_model_fit):
#         self.params = full_model_fit.params
#         self.model = full_model_fit.model  # SARIMAX model structure
#         self.filter_results = full_model_fit.filter_results  # needed for prediction
#         self.data = full_model_fit.data  # optional, safer to keep

#     def predict(self, *args, **kwargs):
#         return self.filter_results.predict(*args, **kwargs)

#     def forecast(self, *args, **kwargs):
#         return self.filter_results.forecast(*args, **kwargs)

#     @staticmethod
#     def params_match(model, param_p, param_d, param_q, param_s):
#         order = getattr(model, 'order', None)
#         seasonal_order = getattr(model, 'seasonal_order', None)

#         if order is None or seasonal_order is None:
#             return False

#         p, d, q = order
#         sp, sd, sq, ss = seasonal_order

#         # Note: assuming seasonal p, d equals param_p, param_d from your earlier code,
#         # adjust if seasonal params differ
#         return (p == param_p and d == param_d and q == param_q and
#                 sp == param_p and sd == param_d and sq == param_s and ss == param_s)

#     @classmethod
#     def from_full_model(cls, full_model_fit):
#         import copy
#         # shallow copy to avoid modifying original fit
#         minimized_fit = copy.copy(full_model_fit)

#         # remove heavy attributes to minimize file size
#         attrs_to_remove = [
#             'cache', '_cache', 'states', 'state_cov', 'design', 'obs', 'transition', 
#             'selection', 'state_names', 'loglikeobs', 'presample', 'kalman_gain'
#         ]
#         for attr in attrs_to_remove:
#             if hasattr(minimized_fit, attr):
#                 setattr(minimized_fit, attr, None)

#         return cls(minimized_fit)

# === SARIMA MODEL TRAINING ===
# models_sarima = {}

for g, modeling_data in modeling_data_groups.items():
    model_id = hash_group(g)

    param_p = model_params.loc[g]['param_p']
    param_d = model_params.loc[g]['param_d']
    param_q = model_params.loc[g]['param_q']
    param_s = model_params.loc[g]['param_s']

    param_q = min(param_q, MAX_Q)

    seasonal_q = find_seasonal_q(param_q, param_s)
    order = (param_s-1 if param_p >= param_s else param_p, param_d, param_q)
    seasonal_order = (param_s, param_d, seasonal_q, param_s)

    sarima_file = make_sarima_filename(model_id, order, seasonal_order)

    retrain_sarima = True
    if os.path.exists(sarima_file):
        try:
            loaded_sarima = SARIMAXResults.load(sarima_file)
            saved_order = loaded_sarima.model.order
            saved_seasonal_order = loaded_sarima.model.seasonal_order

            if saved_order == order and saved_seasonal_order == seasonal_order:
                # models_sarima[g] = loaded_sarima
                print(f"[{g}] Loaded existing SARIMA model from {sarima_file}")
                retrain_sarima = False
            else:
                print(saved_order, order, saved_seasonal_order, seasonal_order)
                # Backup old file with old params in filename
                old_order_str = '_'.join(map(str, saved_order))
                old_seasonal_order_str = '_'.join(map(str, saved_seasonal_order))
                backup_old_model(sarima_file)
            # test predict
            # print(loaded_sarima.forecast(len(modeling_data['test'])))
            tmp_predict = loaded_sarima.predict(start=modeling_data['test'].iloc[0].name, end=modeling_data['test'].iloc[-1].name, dynamic=True)
            test_predicts[g]['sarima'] = tmp_predict
            
        except Exception as e:
            print(f"[{g}] Failed loading SARIMA model: {e}. Will retrain.")

    if retrain_sarima:
        print(f"[{g}] Training SARIMA{order} x {seasonal_order}...")
        try:
            model_sarima = SARIMAX(
                modeling_data['train'][col_target],
                order=order,
                seasonal_order=seasonal_order
            ).fit()
        except np.linalg.LinAlgError as e:
            print(f"[{g}] LinAlgError caught: {e}. Retrying with relaxed constraints...")
            model_sarima = SARIMAX(
                modeling_data['train'][col_target],
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit()
        # minimal_model = MinimalSARIMAXModel(model_sarima)
        # # Save minimal model
        # with open(sarima_file, 'wb') as f:
        #     pickle.dump(minimal_model, f)
        # models_sarima[g] = minimal_model
        model_sarima.save(sarima_file)
        # models_sarima[g] = model_sarima
        print(f"[{g}] SARIMA model trained and saved as {sarima_file}")
        # display(models_sarima[g].summary())



[('Beras Medium', 'Banten')] Loaded existing SARIMA model from ./models/model_sarima_komoditas_28_provinsi_16_order_4_1_2_seasonal_7_1_3_7.msgpack
2025-01-09    12884.214416
2025-01-10    12841.774781
2025-01-11    12884.013290
2025-01-12    12837.161507
2025-01-13    12950.382472
                  ...     
2025-06-09    13351.816636
2025-06-10    13369.410244
2025-06-11    13373.035386
2025-06-12    13389.651469
2025-06-13    13364.408742
Freq: D, Name: predicted_mean, Length: 156, dtype: float64
[('Beras Medium', 'D.I Yogyakarta')] Loaded existing SARIMA model from ./models/model_sarima_komoditas_28_provinsi_14_order_6_1_3_seasonal_7_1_4_7.msgpack
2025-01-09    12990.782130
2025-01-10    12974.467470
2025-01-11    12982.745418
2025-01-12    12969.745772
2025-01-13    12983.873340
                  ...     
2025-06-09    13307.552544
2025-06-10    13323.974304
2025-06-11    13331.777336
2025-06-12    13324.519794
2025-06-13    13306.419349
Freq: D, Name: predicted_mean, Length: 156, d

### **Model SARIMA Evaluation**

In [53]:
# %%script false --no-raise-error
for g, modeling_data in modeling_data_groups.items():
    model_id = hash_group(g)

    param_p = model_params.loc[g]['param_p']
    param_d = model_params.loc[g]['param_d']
    param_q = model_params.loc[g]['param_q']
    param_s = model_params.loc[g]['param_s']

    param_q = min(param_q, MAX_Q)

    seasonal_q = find_seasonal_q(param_q, param_s)
    order = (param_s-1 if param_p >= param_s else param_p, param_d, param_q)
    seasonal_order = (param_s, param_d, seasonal_q, param_s)
    sarima_file = make_sarima_filename(model_id, order, seasonal_order)
    loaded_sarima = SARIMAXResults.load(sarima_file)
    
    tmp_predict = loaded_sarima.predict(start=modeling_data['test'].iloc[0].name, end=modeling_data['test'].iloc[-1].name, dynamic=True)
    test_predicts[g]['sarima'] = tmp_predict


In [54]:
# %%script false --no-raise-error
for g, modeling_data in modeling_data_groups.items():
    tmp_eval = get_eval_smoothing(modeling_data['test'][col_target], test_predicts[g]['sarima'], title='Model SARIMA')
    for i, c in enumerate(df_group_by_cols):
        tmp_eval.insert(0, c, g[i])
    smoothing_evals = pd.concat([smoothing_evals, tmp_eval]).drop_duplicates()

display(smoothing_evals.loc['Model SARIMA'])

,provinsi,komoditas,MAE,MAPE,RMSE
Model SARIMA,Banten,Beras Medium,324.121330,0.025328,367.871476
Model SARIMA,D.I Yogyakarta,Beras Medium,107.050959,0.008186,135.941820
Model SARIMA,DKI Jakarta,Beras Medium,218.936700,0.016874,245.676968
Model SARIMA,Jawa Barat,Beras Medium,232.452392,0.017850,269.480361
Model SARIMA,Jawa Tengah,Beras Medium,124.682644,0.009497,140.236363
Model SARIMA,Jawa Timur,Beras Medium,169.408682,0.013279,192.472602
Model SARIMA,Banten,Daging Ayam Ras,1691.041681,0.047329,2260.589286
Model SARIMA,D.I Yogyakarta,Daging Ayam Ras,2019.086300,0.067472,2916.001002
Model SARIMA,DKI Jakarta,Daging Ayam Ras,662.269541,0.017402,787.370648
Model SARIMA,Jawa Barat,Daging Ayam Ras,1903.077587,0.054964,2320.566719


# **VI. Best Model Evaluation**

## **Voting Best Model For Each `komoditas` and `Provinsi`**

In [55]:
model_eval_dfs2 = {}
prefilter_models = smoothing_evals.index.unique()
# prefilter_models = ['Model ARIMA', 'Holt Winter - Additive']
for g, modeling_data in modeling_data_groups.items():
    # print('Group:', g)
    model_eval_dfs2[g] = {}
    for ev in eval_cols:
        model_eval_dfs2[g][ev] = smoothing_evals[(smoothing_evals['komoditas'] == g[0]) & (smoothing_evals['provinsi'] == g[1])].loc[prefilter_models][ev].idxmin()
        # print(f'Min of [{ev}] :', smoothing_evals[(smoothing_evals['komoditas'] == g[0]) & (smoothing_evals['provinsi'] == g[1])][ev].idxmin())
    # print()

model_eval_dfs2 = pd.DataFrame(model_eval_dfs2).T
model_eval_dfs2.insert(0, 'Best', model_eval_dfs2[eval_cols].mode(axis=1))

model_eval_dfs2

Best                     MAE  \
Beras Medium          Banten             Model ARIMA             Model ARIMA   
                      D.I Yogyakarta     Model ARIMA             Model ARIMA   
                      DKI Jakarta        Model ARIMA             Model ARIMA   
                      Jawa Barat         Model ARIMA             Model ARIMA   
                      Jawa Tengah     Moving Average          Moving Average   
                      Jawa Timur         Model ARIMA             Model ARIMA   
Daging Ayam Ras       Banten             Model ARIMA             Model ARIMA   
                      D.I Yogyakarta     Holt Linear             Holt Linear   
                      DKI Jakarta        Model ARIMA             Model ARIMA   
                      Jawa Barat      Simple Average          Simple Average   
                      Jawa Tengah        Model ARIMA  Holt Winter - Additive   
                      Jawa Timur         Model ARIMA             Model ARIMA   
Minyak Goreng Kemasan Banten            Model SARIMA            Model SARIMA   
                      D.I Yogyakarta    Model SARIMA            Model SARIMA   
                      DKI Jakarta       Model SARIMA            Model SARIMA   
                      Jawa Barat         Model ARIMA             Model ARIMA   
                      Jawa Tengah       Model SARIMA            Model SARIMA   
                      Jawa Timur        Model SARIMA            Model SARIMA   
Telur Ayam Ras        Banten             Model ARIMA             Model ARIMA   
                      D.I Yogyakarta  Simple Average          Simple Average   
                      DKI Jakarta        Model ARIMA             Model ARIMA   
                      Jawa Barat      Simple Average          Simple Average   
                      Jawa Tengah     Simple Average          Simple Average   
                      Jawa Timur      Simple Average          Simple Average   

                                                MAPE            RMSE  
Beras Medium          Banten             Model ARIMA     Model ARIMA  
                      D.I Yogyakarta     Model ARIMA     Model ARIMA  
                      DKI Jakarta        Model ARIMA     Model ARIMA  
                      Jawa Barat         Model ARIMA     Model ARIMA  
                      Jawa Tengah     Moving Average  Moving Average  
                      Jawa Timur         Model ARIMA     Model ARIMA  
Daging Ayam Ras       Banten             Model ARIMA     Model ARIMA  
                      D.I Yogyakarta     Holt Linear     Holt Linear  
                      DKI Jakarta        Model ARIMA     Model ARIMA  
                      Jawa Barat         Holt Linear  Simple Average  
                      Jawa Tengah        Model ARIMA     Model ARIMA  
                      Jawa Timur         Model ARIMA     Model ARIMA  
Minyak Goreng Kemasan Banten            Model SARIMA    Model SARIMA  
                      D.I Yogyakarta    Model SARIMA    Model SARIMA  
                      DKI Jakarta       Model SARIMA    Model SARIMA  
                      Jawa Barat         Model ARIMA     Model ARIMA  
                      Jawa Tengah       Model SARIMA    Model SARIMA  
                      Jawa Timur        Model SARIMA    Model SARIMA  
Telur Ayam Ras        Banten             Model ARIMA     Model ARIMA  
                      D.I Yogyakarta  Simple Average  Simple Average  
                      DKI Jakarta        Model ARIMA     Model ARIMA  
                      Jawa Barat      Simple Average  Simple Average  
                      Jawa Tengah     Simple Average  Simple Average  
                      Jawa Timur      Simple Average     Model ARIMA

## **Evaluation Metrics Overview For Each Best Model**

In [56]:
model_eval_dfs3 = {}

for g, modeling_data in modeling_data_groups.items():
    # print('Group:', g)
    model_eval_dfs3[g] = {}
    for ev in eval_cols:
        model_eval_dfs3[g][ev] = smoothing_evals[(smoothing_evals['komoditas'] == g[0]) & (smoothing_evals['provinsi'] == g[1])].loc[model_eval_dfs2.loc[g, 'Best'], ev]
        # print(f'Min of [{ev}] :', smoothing_evals[(smoothing_evals['komoditas'] == g[0]) & (smoothing_evals['provinsi'] == g[1])][ev].idxmin())
    # print()

model_eval_dfs3 = pd.DataFrame(model_eval_dfs3).T
model_eval_dfs3.insert(0, 'Best', model_eval_dfs2['Best'])

model_eval_dfs3

Best          MAE      MAPE  \
Beras Medium          Banten             Model ARIMA    84.348507  0.006596   
                      D.I Yogyakarta     Model ARIMA    62.692953  0.004780   
                      DKI Jakarta        Model ARIMA   151.255983  0.011634   
                      Jawa Barat         Model ARIMA    81.901543  0.006287   
                      Jawa Tengah     Moving Average    89.028944  0.006752   
                      Jawa Timur         Model ARIMA    73.278240  0.005713   
Daging Ayam Ras       Banten             Model ARIMA   950.141291  0.026432   
                      D.I Yogyakarta     Holt Linear  1248.335838  0.039521   
                      DKI Jakarta        Model ARIMA   581.479015  0.015150   
                      Jawa Barat      Simple Average  1239.707911  0.035666   
                      Jawa Tengah        Model ARIMA  1062.326962  0.030973   
                      Jawa Timur         Model ARIMA  1489.202246  0.046287   
Minyak Goreng Kemasan Banten            Model SARIMA   347.232182  0.016730   
                      D.I Yogyakarta    Model SARIMA   346.040789  0.017525   
                      DKI Jakarta       Model SARIMA   144.968672  0.007025   
                      Jawa Barat         Model ARIMA   365.108325  0.018387   
                      Jawa Tengah       Model SARIMA   347.469356  0.017781   
                      Jawa Timur        Model SARIMA   314.158367  0.015892   
Telur Ayam Ras        Banten             Model ARIMA   782.974774  0.027734   
                      D.I Yogyakarta  Simple Average  1313.463536  0.047179   
                      DKI Jakarta        Model ARIMA   669.720329  0.023660   
                      Jawa Barat      Simple Average  1140.022232  0.039804   
                      Jawa Tengah     Simple Average  1106.042385  0.039808   
                      Jawa Timur      Simple Average  1050.588327  0.038548   

                                             RMSE  
Beras Medium          Banten           105.183418  
                      D.I Yogyakarta    83.474617  
                      DKI Jakarta      167.267071  
                      Jawa Barat       101.085079  
                      Jawa Tengah      105.922323  
                      Jawa Timur        97.685103  
Daging Ayam Ras       Banten          1239.852497  
                      D.I Yogyakarta  1704.204603  
                      DKI Jakarta      764.069826  
                      Jawa Barat      1556.258972  
                      Jawa Tengah     1284.773648  
                      Jawa Timur      1854.085536  
Minyak Goreng Kemasan Banten           821.999312  
                      D.I Yogyakarta   467.300946  
                      DKI Jakarta      185.032900  
                      Jawa Barat       386.022214  
                      Jawa Tengah      389.502266  
                      Jawa Timur       349.582744  
Telur Ayam Ras        Banten          1092.189631  
                      D.I Yogyakarta  1625.295676  
                      DKI Jakarta      887.388633  
                      Jawa Barat      1441.027662  
                      Jawa Tengah     1442.008586  
                      Jawa Timur      1381.303466

## **Best Model Popularity**

In [57]:
pd.DataFrame(model_eval_dfs3['Best'].value_counts())

,Best
Model ARIMA,12
Simple Average,5
Model SARIMA,5
Moving Average,1
Holt Linear,1


## **Best Model For Each `komoditas`**

In [58]:
model_eval_dfs3.reset_index(names=df_group_by_cols).groupby(by='komoditas')[['Best']].apply(
    lambda x: pd.Series({
        'Modes': [mode[0] for mode in x.value_counts()[x.value_counts() == x.value_counts().max()].index],
        'Count': x.value_counts().max()
    })
).reset_index()

,komoditas,Modes,Count
0,Beras Medium,[Model ARIMA],5
1,Daging Ayam Ras,[Model ARIMA],4
2,Minyak Goreng Kemasan,[Model SARIMA],5
3,Telur Ayam Ras,[Simple Average],4


## **Best Model For Each `provinsi`**

In [59]:
model_eval_dfs3.reset_index(names=df_group_by_cols).groupby(by='provinsi')[['Best']].apply(
    lambda x: pd.Series({
        'Modes': [mode[0] for mode in x.value_counts()[x.value_counts() == x.value_counts().max()].index],
        'Count': x.value_counts().max()
    })
).reset_index()

,provinsi,Modes,Count
0,Banten,[Model ARIMA],3
1,D.I Yogyakarta,"[Holt Linear, Model ARIMA, Model SARIMA, Simpl...",1
2,DKI Jakarta,[Model ARIMA],3
3,Jawa Barat,"[Model ARIMA, Simple Average]",2
4,Jawa Tengah,"[Model ARIMA, Model SARIMA, Moving Average, Si...",1
5,Jawa Timur,[Model ARIMA],2


# **VII. Best Model Saving**

## **Save Best Model For Each Group**

In [60]:
def convert_arima_model(old_filename, new_filename):
    # Load the ARIMAResults object from file
    # Note: ARIMAResults.load expects a pickle file, so if your old file is actually msgpack,
    # this won't work directly unless your msgpack file was created in a compatible way.
    # If your old file is a pickle, use this directly:
    model = ARIMAResults.load(old_filename)

    # Save as pickle .pkl file
    with open(new_filename, 'wb') as f:
        pickle.dump(model, f)
    
    print(f"Converted and saved model from {old_filename} to {new_filename}")

def convert_sarima_model(old_filename, new_filename):
    # Load the SARIMAXResults object from file
    # Note: SARIMAXResults.load expects a pickle file, so if your old file is actually msgpack,
    # this won't work directly unless your msgpack file was created in a compatible way.
    # If your old file is a pickle, use this directly:
    model = SARIMAXResults.load(old_filename)

    # Save as pickle .pkl file
    with open(new_filename, 'wb') as f:
        pickle.dump(model, f)
    
    print(f"Converted and saved model from {old_filename} to {new_filename}")


In [61]:
for g, modeling_data in modeling_data_groups.items():
    filename = f'./best_models/best_model_{hash_group(g)}.pkl'
    best_model = None
    if model_eval_dfs3.loc[g, 'Best'] == 'Model ARIMA':
        best_model = models_arima[g]
        order = (model_params.loc[g, 'param_p'], model_params.loc[g, 'param_d'], min(model_params.loc[g, 'param_q'], MAX_Q))
        arima_file = make_arima_filename(hash_group(g), order)
        if os.path.exists(arima_file):
            convert_arima_model(arima_file, filename)
            continue
        else:
            print(f'[{g}] Hello? {arima_file}')
    elif model_eval_dfs3.loc[g, 'Best'] == 'Model SARIMA':
        # best_model = models_sarima[g]
        best_model = True
        order = (model_params.loc[g, 'param_s']-1 if model_params.loc[g, 'param_p'] >= model_params.loc[g, 'param_s'] else model_params.loc[g, 'param_p'], model_params.loc[g, 'param_d'], min(model_params.loc[g, 'param_q'], MAX_Q))
        s_order = (model_params.loc[g, 'param_s'], model_params.loc[g, 'param_d'], min(find_seasonal_q(model_params.loc[g, 'param_q'], model_params.loc[g, 'param_s']), MAX_Q), model_params.loc[g, 'param_s'])
        sarima_file = make_sarima_filename(hash_group(g), order, s_order)
        if os.path.exists(sarima_file):
            convert_sarima_model(sarima_file, filename)
            continue
        else:
            print(f'[{g}] Hello? {sarima_file}')
    else:
        if model_eval_dfs3.loc[g, 'Best'] == 'Simple Average':
            best_model = models_simple_average[g]
        elif model_eval_dfs3.loc[g, 'Best'] == 'Moving Average':
            best_model = models_moving_average[g]
        elif model_eval_dfs3.loc[g, 'Best'] == 'Simple Exponential Smoothing':
            best_model = models_simple_exp_smoothing[g]
        elif model_eval_dfs3.loc[g, 'Best'] == 'Holt Linear':
            best_model = models_holt_linear[g]
        elif model_eval_dfs3.loc[g, 'Best'] == 'Holt Winter - Additive':
            best_model = models_holt_winter_additive[g]
        elif model_eval_dfs3.loc[g, 'Best'] == 'Holt Winter - Multiplicative':
            best_model = models_holt_winter_multiplicative[g]

        if best_model != None:
            with open(filename, 'wb') as f:
                pickle.dump(best_model, f)
            continue

    if best_model == None:
        print(f'[{g}] Error Determining Best Model:', model_eval_dfs3.loc[g, 'Best'])
        continue


Converted and saved model from ./models/model_arima_komoditas_28_provinsi_16_order_4_1_2.msgpack to ./best_models/best_model_komoditas_28_provinsi_16.pkl
Converted and saved model from ./models/model_arima_komoditas_28_provinsi_14_order_6_1_3.msgpack to ./best_models/best_model_komoditas_28_provinsi_14.pkl
Converted and saved model from ./models/model_arima_komoditas_28_provinsi_11_order_7_1_2.msgpack to ./best_models/best_model_komoditas_28_provinsi_11.pkl
Converted and saved model from ./models/model_arima_komoditas_28_provinsi_12_order_5_1_2.msgpack to ./best_models/best_model_komoditas_28_provinsi_12.pkl
Converted and saved model from ./models/model_arima_komoditas_28_provinsi_15_order_5_1_2.msgpack to ./best_models/best_model_komoditas_28_provinsi_15.pkl
Converted and saved model from ./models/model_arima_komoditas_35_provinsi_16_order_4_0_7.msgpack to ./best_models/best_model_komoditas_35_provinsi_16.pkl
Converted and saved model from ./models/model_arima_komoditas_35_provinsi_11

In [62]:
wrapper = ModelWrapper.load_models_from_directory("best_models")

Loaded model for Beras Medium - DKI Jakarta
Loaded model for Beras Medium - Jawa Barat
Loaded model for Beras Medium - Jawa Tengah
Loaded model for Beras Medium - D.I Yogyakarta
Loaded model for Beras Medium - Jawa Timur
Loaded model for Beras Medium - Banten
Loaded model for Daging Ayam Ras - DKI Jakarta
Loaded model for Daging Ayam Ras - Jawa Barat
Loaded model for Daging Ayam Ras - Jawa Tengah
Loaded model for Daging Ayam Ras - D.I Yogyakarta
Loaded model for Daging Ayam Ras - Jawa Timur
Loaded model for Daging Ayam Ras - Banten
Loaded model for Telur Ayam Ras - DKI Jakarta
Loaded model for Telur Ayam Ras - Jawa Barat
Loaded model for Telur Ayam Ras - Jawa Tengah
Loaded model for Telur Ayam Ras - D.I Yogyakarta
Loaded model for Telur Ayam Ras - Jawa Timur
Loaded model for Telur Ayam Ras - Banten
Loaded model for Minyak Goreng Kemasan - DKI Jakarta
Loaded model for Minyak Goreng Kemasan - Jawa Barat
Loaded model for Minyak Goreng Kemasan - Jawa Tengah
Loaded model for Minyak Goreng K